In [ ]:
# Load env variables

from dotenv import load_dotenv
load_dotenv()

# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

True

In [30]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [41]:
# Dataset generation function
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex",
    "solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
# To properly parse the JSON response, we'll use prefilling and stop sequences - still works on Haiku

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [42]:
# Testing the Dataset Generation
# Generate the dataset
dataset = generate_dataset()
print(dataset)

[{'task': "Parse an AWS CloudFormation template and extract all resource logical IDs that have a type of 'AWS::Lambda::Function'", 'format': 'python', 'solution_criteria': 'Function should accept a CloudFormation template (dict or JSON string), iterate through Resources section, filter by Lambda function type, and return a list of logical IDs. Should handle missing Resources key gracefully.'}, {'task': "Create a JSON policy document that allows an IAM user to perform read-only operations (GetObject, ListBucket) on a specific S3 bucket named 'my-data-bucket'", 'format': 'json', 'solution_criteria': 'Valid JSON policy with proper Statement array, Effect set to Allow, correct S3 action names, appropriate Resource ARNs for the bucket and its objects, and Principal element if needed for a resource-based policy.'}, {'task': 'Create a regex pattern to validate AWS S3 bucket names according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens, must start and end with lette

In [43]:
# Saving the Dataset - write it to 'dataset.json'
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [34]:
# Implementing a Model Grader
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [35]:
# This function takes a test case and merges it with our prompt template
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [36]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [37]:
# This function orchestrates running a single test case and grading the result
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Integrating Grading into Your Workflow
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [38]:
from statistics import mean

# This function coordinates the entire evaluation process
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # cCalculate an average score across all test cases
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

# This function processes every test case in our dataset and collects all the results into a single list.

In [39]:
# Running the evaluation
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


Average score: 8.5


In [40]:
# Examining the Results
# The evaluation returns a structured JSON array where each object represents one test case result

print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n  \"FunctionName\": \"S3EventProcessor\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-execution-role\",\n  \"Handler\": \"index.lambda_handler\",\n  \"Timeout\": 30,\n  \"MemorySize\": 256,\n  \"Description\": \"Lambda function that processes S3 bucket events\",\n  \"Environment\": {\n    \"Variables\": {}\n  },\n  \"Code\": {\n    \"ZipFile\": \"def lambda_handler(event, context):\\n    for record in event['Records']:\\n        bucket = record['s3']['bucket']['name']\\n        key = record['s3']['object']['key']\\n        print(f'Processing object {key} from bucket {bucket}')\\n    return {'statusCode': 200, 'body': 'Success'}\"\n  },\n  \"Architectures\": [\n    \"x86_64\"\n  ],\n  \"EphemeralStorage\": {\n    \"Size\": 512\n  },\n  \"EventInvokeConfig\": {\n    \"MaximumEventAge\": 3600,\n    \"MaximumRetryAttempts\": 2\n  }\n}\n",
    "test_case": {
      "task": "Create a JSON configuration object for an AWS Lambda f